# SOX2 temporal-resolution figure replot

This notebook uses the deposited final correlation curves and statistics. The publication plotting cell is copied verbatim, but the upstream trace-level correlation and permutation analysis is not recomputed because the three trace CSV inputs are not in the current release candidate.


In [ ]:
from pathlib import Path
import os

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "analyses").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the Gao2026 code root")

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
GAO2026_DATA_ROOT = Path(
    os.environ.get("GAO2026_DATA_ROOT", REPO_ROOT / "data")
).expanduser().resolve()
GAO2026_OUTPUT_ROOT = Path(
    os.environ.get("GAO2026_OUTPUT_ROOT", REPO_ROOT / "outputs")
).expanduser().resolve()

DATA_DIR = GAO2026_DATA_ROOT / "06_SOX2_temporal_resolution"
OUTPUT_SUMMARY = GAO2026_OUTPUT_ROOT / "06_SOX2_temporal_resolution" / "output_summary"
OUTPUT_SUMMARY.mkdir(parents=True, exist_ok=True)

summary = pd.read_csv(DATA_DIR / "output_summary/SOX2_temporal_resolution_summary.csv")
curves = pd.read_csv(DATA_DIR / "output_summary/SOX2_cross_correlation_curves.csv")

results = {}
for row in summary.to_dict("records"):
    label = row["condition"]
    curve = curves.loc[curves["condition"] == label].sort_values("lag_frames")
    if curve.empty:
        raise ValueError(f"Missing cross-correlation curve: {label}")
    results[label] = {
        "cross_corr": pd.DataFrame({
            "mean": curve["cross_correlation_mean"].to_numpy(float),
            "low": curve["cross_correlation_ci_low"].to_numpy(float),
            "high": curve["cross_correlation_ci_high"].to_numpy(float),
        }),
        "dt_min": float(row["sampling_interval_min"]),
        "N_cells": int(row["N_cells"]),
        "LLI_median": float(row["LLI_median"]),
        "LLI_ci_low": float(row["LLI_ci_low"]),
        "LLI_ci_high": float(row["LLI_ci_high"]),
        "perm_p_two": float(row["permutation_p_two_sided"]),
        "perm_q_two_BH": float(row["permutation_q_two_sided_BH"]),
    }

lag_arrays = [
    curves.loc[curves["condition"] == label]
    .sort_values("lag_frames")["lag_frames"].to_numpy(int)
    for label in summary["condition"]
]
LAGS = lag_arrays[0]
if not all(np.array_equal(LAGS, values) for values in lag_arrays[1:]):
    raise ValueError("Stored conditions do not share a common frame-lag grid")

mpl.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 150,
})

def save_and_show(figure, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output_path, format="pdf", bbox_inches="tight")
    display(figure)
    plt.close(figure)
    print(f"Saved: {output_path}")

display(summary)


In [ ]:
FONT_SIZE = 14

crosscorr_panels = [
    ("SOX2_1min", "1 min interval"),
    ("SOX2_30s", "30 s interval"),
]

lli_panels = [
    ("SOX2_30s", "SOX2      \n(30 s interval)"),
    ("SOX2_1min", "SOX2       \n(1 min interval)"),
]

figure, axes = plt.subplots(
    1,
    3,
    figsize=(15, 3.3),
    gridspec_kw={
        "width_ratios": [1.0, 1.0, 1.65],
    },
)

crosscorr_y_min = np.inf
crosscorr_y_max = -np.inf


# ------------------------------------------------------------
# Cross-correlation plots
# ------------------------------------------------------------
for axis, (label, interval_label) in zip(
    axes[:2],
    crosscorr_panels,
):
    result = results[label]
    curve = result["cross_corr"]
    lag_minutes = LAGS * result["dt_min"]

    crosscorr_y_min = min(
        crosscorr_y_min,
        np.nanmin(curve["low"]),
    )
    crosscorr_y_max = max(
        crosscorr_y_max,
        np.nanmax(curve["high"]),
    )

    axis.fill_between(
        lag_minutes,
        curve["low"],
        curve["high"],
        color="tab:blue",
        alpha=0.18,
        linewidth=0,
    )
    axis.plot(
        lag_minutes,
        curve["mean"],
        color="tab:blue",
        linewidth=2.5,
        label="SOX2",
    )

    axis.axhline(
        0,
        color="black",
        linewidth=1,
        linestyle="--",
    )
    axis.axvline(
        0,
        color="black",
        linewidth=1,
        linestyle=":",
    )

    maximum_lag_min = np.max(np.abs(lag_minutes))

    axis.set_xlim(
        -maximum_lag_min,
        maximum_lag_min,
    )
    axis.set_xticks(
        np.linspace(
            -maximum_lag_min,
            maximum_lag_min,
            5,
        )
    )

    axis.set_xlabel(
        r"Lag $\tau$ (min)",
        fontsize=FONT_SIZE,
    )
    axis.set_title(
        "Mean cross-correlation ±95% CI\n"
        f"{interval_label}",
        fontsize=FONT_SIZE,
    )

    axis.grid(
        axis="both",
        linestyle=":",
        linewidth=0.8,
        alpha=0.5,
    )
    axis.legend(
        frameon=False,
        loc="upper right",
        fontsize=FONT_SIZE,
    )
    axis.tick_params(labelsize=FONT_SIZE)

axes[0].set_ylabel(
    "Cross-corr (baseline-corrected)",
    fontsize=FONT_SIZE,
)

y_range = crosscorr_y_max - crosscorr_y_min
y_padding = 0.08 * y_range

for axis in axes[:2]:
    axis.set_ylim(
        crosscorr_y_min - y_padding,
        crosscorr_y_max + y_padding,
    )


# ------------------------------------------------------------
# Combined LLI plot
# Positive displayed LLI = MCP leads
# ------------------------------------------------------------
axis = axes[2]
y_positions = np.arange(len(lli_panels))

displayed_values = []

for y_position, (label, row_label) in zip(
    y_positions,
    lli_panels,
):
    result = results[label]

    # Raw LLI:
    #   positive = SOX2/SNAP leads
    #
    # Displayed LLI:
    #   positive = MCP leads
    displayed_median = -result["LLI_median"]
    displayed_low = -result["LLI_ci_high"]
    displayed_high = -result["LLI_ci_low"]

    displayed_values.append(
        (
            displayed_low,
            displayed_median,
            displayed_high,
        )
    )

    axis.errorbar(
        displayed_median,
        y_position,
        xerr=[
            [displayed_median - displayed_low],
            [displayed_high - displayed_median],
        ],
        fmt="o",
        markersize=6,
        color="black",
        ecolor="black",
        elinewidth=1.5,
        capsize=3,
        capthick=1.2,
    )

    axis.text(
        0.98,
        y_position,
        (
            f"n={result['N_cells']}, "
            f"p={result['perm_p_two']:.3g}, "
            f"q={result['perm_q_two_BH']:.3g}"
        ),
        transform=axis.get_yaxis_transform(),
        horizontalalignment="right",
        verticalalignment="center",
        fontsize=FONT_SIZE,
    )

axis.axvline(
    0,
    color="black",
    linewidth=1,
    linestyle="--",
)

all_lows = [value[0] for value in displayed_values]
all_highs = [value[2] for value in displayed_values]

x_min = min(-0.10, min(all_lows) - 0.03)
x_max = max(0.20, max(all_highs) + 0.12)

axis.set_xlim(x_min, x_max)
axis.set_ylim(
    len(lli_panels) - 0.5,
    -0.5,
)

axis.set_yticks(y_positions)
axis.set_yticklabels(
    [row_label for _, row_label in lli_panels],
    fontsize=FONT_SIZE,
)

axis.set_xlabel(
    "LLI (median) ± bootstrap 95% CI | "
    "positive: MCP leads",
    fontsize=FONT_SIZE,
)
axis.set_title(
    "LLI + time-shift permutation",
    fontsize=FONT_SIZE,
)

axis.grid(
    axis="x",
    linestyle=":",
    linewidth=0.8,
    alpha=0.5,
)
axis.tick_params(axis="x", labelsize=FONT_SIZE)

figure.tight_layout(w_pad=2.5)

save_and_show(
    figure,
    OUTPUT_SUMMARY
    / "SOX2_1min_30s_cross_correlation_and_LLI.pdf",
)